# Track 2: Atlas-Free Coordinate GNN — Google Colab Training

**Goal**: Train `CoordGNN` on raw MNI peak coordinates (no atlas consulted).  
**Hardware**: A100 40GB — Runtime → Change runtime type → A100 GPU  
**Output**: `best_coord_gnn.pt` and `last_coord_gnn.pt` saved to Google Drive

### Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Run all cells top-to-bottom.

### Resuming after a session crash
Re-run all cells from the top. The graph cache lives on Drive — it won't be rebuilt.

## 0 — Runtime check

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        'No GPU detected.\n'
        'Go to Runtime → Change runtime type → select A100 GPU, then re-run.'
    )
print(gpu_info.stdout)

## 1 — Install dependencies

In [ ]:
import torch, sys
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')
torch_tag = torch.__version__.split('+')[0]
print(f'torch={torch_tag}  cuda={cuda_tag}')

In [ ]:
# Minimal installs — coordinates are already in a parquet, text embeddings are
# pre-computed .pt files, so nibabel/nimare/transformers are NOT needed here.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow', 'pandas', 'matplotlib', 'tqdm', 'umap-learn'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'nilearn'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric'], check=True)

pyg_url = f'https://data.pyg.org/whl/torch-{torch_tag}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_scatter', 'torch_sparse', 'torch_cluster',
                '-f', pyg_url], check=True)

print('Dependencies installed.')

## 2 — Clone the repo

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    !git clone --branch neurovlm_gnn https://github.com/neurovlm/neurovlm.git {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes.')
    !git -C {REPO_DIR} pull

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print('sys.path entry:', src_dir)

In [ ]:
# Patch neurovlm/__init__.py to lazy-load so heavy optional deps
# (NeuroAutoEncoder, Specter, etc.) are not imported unless needed.
init_path = REPO_DIR / 'src' / 'neurovlm' / '__init__.py'
init_path.write_text(
    'from __future__ import annotations\n\n\n'
    'def __getattr__(name: str):\n'
    '    if name == "NeuroVLM":\n'
    '        from .core import NeuroVLM\n'
    '        return NeuroVLM\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n'
)
print('Patched:', init_path)

## 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR      = Path('/content/drive/MyDrive/neurovlm_track2')
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints/coord_gnn'
CACHE_DIR      = DRIVE_DIR / 'data/coord_graphs'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Checkpoints → {CHECKPOINT_DIR}')
print(f'Graph cache → {CACHE_DIR}')

best_ckpt = CHECKPOINT_DIR / 'best_coord_gnn.pt'
last_ckpt = CHECKPOINT_DIR / 'last_coord_gnn.pt'
if last_ckpt.exists():
    print(f'\nResume checkpoint found: {last_ckpt}')
    print('Training will pick up from the last saved epoch.')
else:
    print('\nNo prior checkpoint — starting fresh.')

## 4 — Hyperparameters

In [ ]:
CFG = dict(
    K            = 7,      # KNN neighbors per node
    MAX_DIST_MM  = 30.0,   # prune edges beyond this mm distance
    HIDDEN       = 128,    # hidden dim per attention head
    HEADS        = 8,
    DROPOUT      = 0.1,
    N_EPOCHS     = 200,
    BATCH_SIZE   = 128,    # A100 handles 128 comfortably; reduce to 64 if OOM
    LR_GNN       = 1e-4,
    LR_PROJ      = 1e-5,
    WARMUP_EPOCHS= 15,
    TEMPERATURE  = 0.07,
    VAL_INTERVAL = 5,
    SEED         = 42,
)
print('Config:', CFG)

## 5 — Load coordinates & statistics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from neurovlm.retrieval_resources import _load_pubmed_coordinates
from neurovlm.gnn.coord_graph import normalize_coords, coords_to_graph

print('Loading peak coordinates …')
coords_df = _load_pubmed_coordinates()
print(f'  Shape: {coords_df.shape}  columns: {list(coords_df.columns)}')

print('\nFirst 3 papers:')
for pmid, grp in list(coords_df.groupby('pmid'))[:3]:
    print(f'  PMID {pmid}: {len(grp)} peaks | '
          f'x∈[{grp.x.min():.0f},{grp.x.max():.0f}] '
          f'y∈[{grp.y.min():.0f},{grp.y.max():.0f}] '
          f'z∈[{grp.z.min():.0f},{grp.z.max():.0f}]')

In [ ]:
peak_counts = coords_df.groupby('pmid').size()
print(f'Peak counts across {len(peak_counts):,} papers:')
print(f'  min={peak_counts.min()}  max={peak_counts.max()}')
print(f'  mean={peak_counts.mean():.1f}  median={peak_counts.median():.0f}')
print(f'  p5={peak_counts.quantile(0.05):.0f}  p95={peak_counts.quantile(0.95):.0f}')
print(f'  Papers with <3 peaks: {(peak_counts<3).sum()} ({100*(peak_counts<3).mean():.1f}%)')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(peak_counts.clip(upper=100), bins=50, color='steelblue', edgecolor='white', lw=0.5)
ax.set_xlabel('Peaks per paper'); ax.set_ylabel('# papers')
ax.set_title('Peak count distribution (clipped at 100)')
plt.tight_layout(); plt.show()

In [ ]:
# Normalization and duplicate check
coords_arr = coords_df[['x','y','z']].values
norm_arr   = normalize_coords(coords_arr)
print(f'Normalized range: [{norm_arr.min():.4f}, {norm_arr.max():.4f}]')
print(f'Outliers clipped: {(np.abs(norm_arr) > 1.05).any(axis=1).sum()}')

dup_papers, max_dups = 0, 0
for _, grp in coords_df.groupby('pmid'):
    raw = grp[['x','y','z']].values
    n_dups = len(raw) - len(np.unique(raw, axis=0))
    if n_dups > 0:
        dup_papers += 1
        max_dups = max(max_dups, n_dups)
print(f'Papers with duplicate peaks: {dup_papers}  (max dups in one paper: {max_dups})')

In [ ]:
# k sensitivity: avg degree and % edges pruned by distance at k=5,7,10
import random
sample_pmids = random.sample(list(coords_df['pmid'].unique()), 100)
sample_df = coords_df[coords_df['pmid'].isin(sample_pmids)]

print('k  | avg_degree | pct_filtered')
print('---+------------+-------------')
for k_val in [5, 7, 10]:
    degrees, frac_filt = [], []
    for pmid, grp in sample_df.groupby('pmid'):
        raw  = np.unique(grp[['x','y','z']].values, axis=0)
        norm = normalize_coords(raw)
        g    = coords_to_graph(norm, k=k_val, max_dist_mm=30.0)
        n, e = g.x.shape[0], g.edge_index.shape[1]
        max_e = n * min(k_val, n-1) if n > 1 else 0
        degrees.append(e / max(n, 1))
        if max_e > 0: frac_filt.append(1 - e/max_e)
    print(f' {k_val:2d} | {np.mean(degrees):10.2f} | {np.mean(frac_filt)*100:10.1f}%')
print('\nTarget: avg_degree ∈ [4, 12] and pct_filtered < 20%')

## 6 — Dataset

In [ ]:
from neurovlm.data import load_dataset, load_latent

print('Loading SPECTER text embeddings …')
text_latents = load_latent('pubmed_text')
pubmed_df    = load_dataset('pubmed_text')

if isinstance(text_latents, tuple):
    text_tensor, pmids_arr = text_latents
    unique_pmids = np.asarray(pmids_arr).astype(str)
elif isinstance(text_latents, dict):
    pmid_list    = list(text_latents.keys())
    text_tensor  = torch.stack([torch.tensor(text_latents[p], dtype=torch.float32)
                                for p in pmid_list])
    unique_pmids = np.array([str(p) for p in pmid_list])
else:
    text_tensor  = text_latents if isinstance(text_latents, torch.Tensor) \
                   else torch.tensor(text_latents)
    unique_pmids = pubmed_df['pmid'].astype(str).values[:len(text_tensor)] \
                   if 'pmid' in pubmed_df.columns \
                   else np.arange(len(text_tensor)).astype(str)

print(f'  text_tensor  : {text_tensor.shape}')
print(f'  unique_pmids : {len(unique_pmids):,}')

In [ ]:
from neurovlm.gnn.coord_dataset import CoordGraphDataset

print(f'Building CoordGraphDataset (k={CFG["K"]}, max_dist={CFG["MAX_DIST_MM"]}mm) …')
print(f'  Cache dir: {CACHE_DIR}')
print('  Graphs already on Drive will be reused — only missing ones are built.')

ds = CoordGraphDataset(
    coords_df       = coords_df,
    text_embeddings = text_tensor,
    unique_pmids    = unique_pmids,
    cache_dir       = str(CACHE_DIR),
    k               = CFG['K'],
    max_dist_mm     = CFG['MAX_DIST_MM'],
)
print(f'Dataset size: {len(ds):,} papers')

In [ ]:
# Verify PyG batching
from torch_geometric.loader import DataLoader
probe = next(iter(DataLoader(ds, batch_size=4, shuffle=False)))
print('PyG batch check (batch_size=4):')
print(f'  x.shape          = {probe.x.shape}    ← (total_nodes, 5)')
print(f'  edge_index.shape = {probe.edge_index.shape}')
print(f'  edge_attr.shape  = {probe.edge_attr.shape}    ← (E, 4)')
print(f'  y.shape          = {probe.y.shape}')
assert probe.x.shape[1] == 5,     'node dim must be 5'
assert probe.edge_attr.shape[1] == 4, 'edge dim must be 4'
print('✓ All shapes correct')

In [ ]:
torch.manual_seed(CFG['SEED'])
train_ds, val_ds, test_ds = ds.split(val_frac=0.1, test_frac=0.1, seed=CFG['SEED'])
print(f'Split: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

## 7 — Model

In [ ]:
from torch_geometric.data import Data, Batch
from neurovlm.gnn.coord_model import CoordGNN
from neurovlm.gnn.model import TextProjHead

brain_encoder = CoordGNN(
    in_dim=5, hidden=CFG['HIDDEN'], heads=CFG['HEADS'],
    out_dim=384, dropout=CFG['DROPOUT'],
)
text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=384)

n_brain = brain_encoder.count_parameters()
n_text  = sum(p.numel() for p in text_proj.parameters())
print(f'CoordGNN params     : {n_brain:,}')
print(f'TextProjHead params : {n_text:,}')

assert 500_000 <= n_brain <= 5_000_000, f'Param count {n_brain:,} out of [500K, 5M]'

dummy = Data(x=torch.randn(10,5),
             edge_index=torch.tensor([[0,1,2],[1,2,0]], dtype=torch.long),
             edge_attr=torch.randn(3,4))
b = Batch.from_data_list([dummy, dummy])
out = brain_encoder(b.x, b.edge_index, b.edge_attr, b.batch)
assert out.shape == (2, 384)
print(f'✓ Forward pass: (20, 5) → {tuple(out.shape)}')

## 8 — Training

In [ ]:
from neurovlm.gnn.coord_train import CoordTrainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

trainer = CoordTrainer(
    brain_encoder  = brain_encoder,
    text_proj      = text_proj,
    lr_gnn         = CFG['LR_GNN'],
    lr_proj        = CFG['LR_PROJ'],
    batch_size     = CFG['BATCH_SIZE'],
    n_epochs       = CFG['N_EPOCHS'],
    warmup_epochs  = CFG['WARMUP_EPOCHS'],
    temperature    = CFG['TEMPERATURE'],
    device         = DEVICE,
    val_interval   = CFG['VAL_INTERVAL'],
    checkpoint_dir = str(CHECKPOINT_DIR),
    verbose        = True,
)

trainer.fit(train_ds, val_ds)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(trainer.history['train_loss'])
axes[0].set_title('Train Loss (InfoNCE)'); axes[0].set_xlabel('Epoch')

axes[1].plot(trainer.history['val_recall_t2i'], label='t2i')
axes[1].plot(trainer.history['val_recall_i2t'], label='i2t')
axes[1].set_title('Val Recall AUC')
axes[1].set_xlabel(f'Val step (every {CFG["VAL_INTERVAL"]} epochs)'); axes[1].legend()

axes[2].plot(trainer.history['embed_sim'], color='red')
axes[2].axhline(0.95, color='black', ls='--', label='collapse threshold')
axes[2].set_title('embed_sim (collapse monitor)'); axes[2].legend()

plt.tight_layout()
plt.savefig(str(DRIVE_DIR / 'training_curve.png'), dpi=150)
plt.show()

## 9 — Evaluation

In [ ]:
from neurovlm.metrics import recall_at_k, recall_curve

trainer.restore_best()
results = {}

for split_name, split_ds in [('Val', val_ds), ('Test', test_ds)]:
    brain_emb, text_emb = trainer.collect_embeddings(split_ds)
    sim = F.normalize(text_emb, dim=1) @ F.normalize(brain_emb, dim=1).T

    r1    = recall_at_k(sim, 1)
    r5    = recall_at_k(sim, 5)
    r10   = recall_at_k(sim, 10)
    t2i, i2t = recall_curve(text_emb, brain_emb)
    auc   = float((t2i.mean() + i2t.mean()) / 2)

    results[split_name] = dict(r1=r1, r5=r5, r10=r10, auc=auc)
    print(f'\n{split_name} ({len(split_ds)} papers):')
    print(f'  recall@1  = {r1:.4f}')
    print(f'  recall@5  = {r5:.4f}')
    print(f'  recall@10 = {r10:.4f}')
    print(f'  AUC       = {auc:.4f}  (NeuroVLM MLP baseline ≈ 0.81)')

print('\n┌───────────────────────────┬──────────┬────────────┐')
print('│ Model                     │ Test AUC │ Atlas-free │')
print('├───────────────────────────┼──────────┼────────────┤')
print('│ NeuroVLM MLP (baseline)   │ 0.81     │ No         │')
print('│ Track 1 DiFuMo GAT        │ (run it) │ No         │')
print(f'│ Track 2 Coord GNN (ours)  │ {results["Test"]["auc"]:.4f}   │ Yes        │')
print('└───────────────────────────┴──────────┴────────────┘')

## 10 — Attention Analysis

In [ ]:
snapshots = trainer.get_attention_snapshot(test_ds, n_samples=10)

for snap in snapshots:
    n_peaks = snap['node_coords_mni'].shape[0]
    print(f'\nPaper {snap["paper_idx"]} ({n_peaks} peaks):')
    for e in snap['top_edges']:
        dist_mm = float(np.linalg.norm(np.array(e['src_mni']) - np.array(e['dst_mni'])))
        s = [f'{v:.0f}' for v in e['src_mni']]
        d = [f'{v:.0f}' for v in e['dst_mni']]
        print(f'  [{", ".join(s)}] → [{"  ".join(d)}]  attn={e["weight"]:.4f}  dist={dist_mm:.0f}mm')

## 11 — Failure Analysis

In [ ]:
brain_emb, text_emb = trainer.collect_embeddings(test_ds)
sim = F.normalize(text_emb, dim=1) @ F.normalize(brain_emb, dim=1).T

top10_hits = (sim.argsort(dim=1, descending=True)[:, :10] ==
              torch.arange(len(sim)).unsqueeze(1)).any(dim=1)

worst_pos = (~top10_hits).nonzero(as_tuple=True)[0][:20].tolist()
print(f'Papers with recall@10=0: {(~top10_hits).sum()} / {len(sim)}')

worst_loader = DataLoader([test_ds[i] for i in worst_pos], batch_size=1, shuffle=False)
peak_counts_worst = [g.x.shape[0] for g in worst_loader]
print(f'Worst-paper peak counts: mean={np.mean(peak_counts_worst):.1f}  '
      f'min={min(peak_counts_worst)}  max={max(peak_counts_worst)}')

## 12 — UMAP

In [ ]:
try:
    import umap
    brain_emb, _ = trainer.collect_embeddings(test_ds)
    emb_np = F.normalize(brain_emb, dim=1).cpu().numpy()

    N = min(500, len(emb_np))
    idx = np.random.default_rng(42).choice(len(emb_np), N, replace=False)
    emb_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(emb_np[idx])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.6, c='steelblue')
    ax.set_title(f'UMAP of CoordGNN brain embeddings ({N} test papers)')
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    plt.tight_layout()
    plt.savefig(str(DRIVE_DIR / 'umap_coord_gnn.png'), dpi=150)
    plt.show()
except ImportError:
    print('umap-learn not available — skipping.')

## 13 — Save artifacts

In [ ]:
import json

with open(DRIVE_DIR / 'training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals]
               for k, vals in trainer.history.items()}, f, indent=2)

with open(DRIVE_DIR / 'eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Files saved to Drive:')
for p in sorted(DRIVE_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(DRIVE_DIR)}  ({p.stat().st_size/1e6:.1f} MB)')